In [ ]:
import tensorflow as tf
import numpy as np

from pathlib import Path

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

In [ ]:
model = tf.keras.models.load_model(
    "../artifacts/model_trainer/resnet50_finetuned.keras",
    custom_objects={
        "preprocess_input":
            tf.keras.applications.resnet50.preprocess_input
    }
)

print("Model loaded successfully")

In [ ]:
DATASET_DIR = Path(
    "../artifacts/data_ingestion/brisc2025/brisc2025/classification_task"
)

train_dir = DATASET_DIR / "train"

print("Train directory:", train_dir)

In [ ]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224, 224),
    batch_size=16,
    shuffle=False
)

print("Classes:", val_ds.class_names)

In [ ]:
val_predictions = model.predict(val_ds)

y_val_pred = np.argmax(
    val_predictions,
    axis=1
)

y_val_true = np.concatenate(
    [labels.numpy() for _, labels in val_ds]
)

print("True labels shape :", y_val_true.shape)
print("Predicted labels shape :", y_val_pred.shape)

In [ ]:
class_names = val_ds.class_names

print(
    classification_report(
        y_val_true,
        y_val_pred,
        target_names=class_names
    )
)

In [ ]:
print("Unique true labels :", np.unique(y_val_true))
print("Unique predicted labels :", np.unique(y_val_pred))

print("True label counts:")
print(np.bincount(y_val_true))

print("Predicted label counts:")
print(np.bincount(y_val_pred))

In [ ]:
class_names = [
    "glioma",
    "meningioma",
    "no_tumor",
    "pituitary"
]

print("Expected classes:", len(class_names))
print("Classes present in true labels:", len(np.unique(y_val_true)))
print("Classes present in predictions:", len(np.unique(y_val_pred)))

#### Observations:
1.  validation dataset contains 1,000 images, but all 1,000 are being assigned label 3 (pituitary).
2. If the validation split were constructed correctly, we should see something approximately like: Unique true labels : [0 1 2 3]
3. Instead, we have [3] So the problem is with how the validation dataset is being created/loaded.
4. True label distribution:

- glioma: 0
- meningioma: 0
- no_tumor: 0
- pituitary: 1000

This is unexpected because the validation data used during model training
contained all four classes.

Therefore, before evaluating the model, we need to investigate how
`validation_split` behaves when `shuffle=False`


#### Investigate the directory ordering

In [ ]:
class_names = sorted([
    folder.name
    for folder in train_dir.iterdir()
    if folder.is_dir()
])

print("Class names:", class_names)

In [ ]:
for class_name in class_names:
    class_dir = train_dir / class_name

    files = [
        file
        for file in class_dir.iterdir()
        if file.is_file()
    ]

    print(
        f"{class_name}: {len(files)} images"
    )

#### Understand why the last 1,000 are pituitary

In [ ]:
total = 0

for class_name in class_names:
    class_dir = train_dir / class_name

    count = len([
        file
        for file in class_dir.iterdir()
        if file.is_file()
    ])

    start = total
    end = total + count - 1

    print(
        f"{class_name}: positions {start} to {end}"
    )

    total += count

#### Observations:
1. There are total 5000 images and 20% is 1000 images.
2. So with shuffle=False, the validation subset corresponds to the last 1,000 positions (4000 → 4999).
3. The positions are entirely inside pituitary → 3543 to 4999 . This is our root cause

#### Investigation Result 1 — `shuffle=False` and Validation Split

The validation dataset was created with:

`validation_split=0.2`

and:

`shuffle=False`

The directory contains class-ordered files, and the final 1,000 files
belong entirely to the `pituitary` class.

Therefore, the validation dataset created in the debug experiment is not
representative of all four classes.

This explains:

`Unique true labels: [3]`

and:

`True label counts: [0, 0, 0, 1000]`

This is a validation dataset construction issue, not evidence that the
fine-tuned model predicts every MRI as pituitary.

#### Recreate the ORIGINAL validation configuration

In [ ]:
recreate_val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224, 224),
    batch_size=16,
    shuffle=True
)

print("Classes:", recreate_val_ds.class_names)

#### Check labels

In [ ]:
y_val_true = np.concatenate(
    [labels.numpy() for _, labels in recreate_val_ds]
)

print("Unique true labels:", np.unique(y_val_true))
print("True label counts:", np.bincount(y_val_true))

#### Investigate the original evaluation code

In [ ]:
val_predictions = model.predict(recreate_val_ds)

y_val_pred = np.argmax(
    val_predictions,
    axis=1
)

y_val_true = np.concatenate(
    [labels.numpy() for _, labels in recreate_val_ds])

#### Understanding the effect of shuffle=True during evaluation

In [ ]:
# First iteration
first_batch_images, first_batch_labels = next(
    iter(recreate_val_ds)
)

# Second iteration
second_batch_images, second_batch_labels = next(
    iter(recreate_val_ds)
)

print(
    "First batch labels :",
    first_batch_labels.numpy()
)

print(
    "Second batch labels:",
    second_batch_labels.numpy()
)

### Observation
When shuffle=True, a new iteration over the validation dataset can produce the samples in a different order. Therefore, if predictions and true labels are collected through separate iterations, their ordering may not correspond

#### Check whether two complete iterations have the same label order

In [ ]:
labels_first = np.concatenate(
    [labels.numpy() for _, labels in recreate_val_ds]
)

labels_second = np.concatenate(
    [labels.numpy() for _, labels in recreate_val_ds]
)

print("First iteration shape :", labels_first.shape)
print("Second iteration shape:", labels_second.shape)

print(
    "Are label orders identical?",
    np.array_equal(labels_first, labels_second)
)

#### Observation
1. Both iterations contain the same 1,000 validation samples in quantity. But with shuffle=True, their order is different.
2. For evaluation, predictions and true labels must come from the same fixed ordering.
3. shuffle=True correctly gives us all four classes, but the order can change between iterations. Therefore, predictions and true labels collected separately may not correspond to the same images.
4. We will freeze one complete validation iteration and use the same batches to obtain both predictions and true labels.
5. Will ensure that every prediction is compared with the correct true label.

#### Freeze one validation iteration

In [ ]:

validation_data = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224, 224),
    batch_size=16,
    shuffle=True
)

print("Classes:", validation_data.class_names)

# Freeze the dataset into memory
val_fixed = list(validation_data)

print("Number of batches:", len(val_fixed))
print("Total images:", sum(
    images.shape[0] for images, labels in val_fixed
))

#### Generate predictions and labels from the SAME batches

In [ ]:
y_val_true = []
y_val_pred = []

for images, labels in val_fixed:

    predictions = model.predict(
        images,
        verbose=0
    )

    predicted_labels = np.argmax(
        predictions,
        axis=1
    )

    y_val_true.extend(
        labels.numpy()
    )

    y_val_pred.extend(
        predicted_labels
    )

y_val_true = np.array(y_val_true)
y_val_pred = np.array(y_val_pred)

print("True labels shape :", y_val_true.shape)
print("Predicted labels shape :", y_val_pred.shape)

In [ ]:
print("Unique true labels :", np.unique(y_val_true))

print("True label counts:")
print(np.bincount(y_val_true))

### calculate the classification report

In [ ]:
class_names = validation_data.class_names

print(
    classification_report(
        y_val_true,
        y_val_pred,
        labels=[0, 1, 2, 3],
        target_names=class_names
    )
)

In [ ]:
cm = confusion_matrix(
    y_val_true,
    y_val_pred,
    labels=[0, 1, 2, 3]
)

print(cm)

In [ ]:
val_accuracy = accuracy_score(
    y_val_true,
    y_val_pred
)

macro_f1 = f1_score(
    y_val_true,
    y_val_pred,
    average="macro"
)

print("Validation Accuracy :", val_accuracy)
print("Validation Macro F1 :", macro_f1)